# Notebook 2: Experimental Designs, Synchronization Methods, and Control Conditions in Hyperscanning Analysis with HyPyP

In this notebook, we explore advanced methodological aspects of hyperscanning analyses. We will:
- Load dyads from a folder structure (or individual files).
- Apply preprocessing (ICA) to remove artifacts.
- Create pseudo-dyads and surrogate data as control conditions.
- Compute and compare inter-brain connectivity metrics for real and control conditions.

In [ ]:
import os
import glob
import pickle
from collections import OrderedDict

import mne
import numpy as np
from scipy import fft

import hypyp.analyses as analyses
import hypyp.viz as viz
import hypyp.prep as prep

# Confirm successful import of libraries
print("Libraries imported successfully.")

## 1. Loading the Preprocessed Data

Instead of reloading the epoch files and performing ICA preprocessing again, we will load the preprocessed data that was saved in the previous notebook. This allows us to skip the time-consuming preprocessing steps and continue directly with our analysis.

In [ ]:
# Load preprocessed data from pickle file
filename = './data/generated/hyperscanning_data.pkl'
#filename = './data/generated/hyperscanning_recorded_data.pkl'

with open(filename, 'rb') as f:
    saved_data = pickle.load(f)

# Extract the cleaned epochs from the loaded data
epo1_clean = saved_data['epo1_clean']
epo2_clean = saved_data['epo2_clean']

# Print summaries for verification
print("Participant 1 Cleaned Epochs:")
print(epo1_clean)
print("\nParticipant 2 Cleaned Epochs:")
print(epo2_clean)

# Store dyad data as an array (for later connectivity computations)
dyad_clean = [epo1_clean.get_data(copy=True), epo2_clean.get_data(copy=True)]

## 2. Creating Control Conditions: Pseudo-Dyads and Surrogate Data

To evaluate whether inter-brain synchrony is due to true interaction or common stimulus effects, we generate:
- **Pseudo-Dyads:** by shuffling epochs within a dyad.
- **Surrogate Data:** by randomizing the phase in frequency domain while preserving amplitude information, then reconstructing the signal via inverse Fourier transform.

These control conditions help us determine whether observed synchrony is genuinely due to interaction between participants.

In [ ]:
# Create a pseudo-dyad by shuffling the epochs for one participant.
pseudo_epo1 = epo1_clean.copy()
pseudo_epo2 = epo2_clean.copy()
# Shuffle epochs along the first axis of the data array
np.random.shuffle(pseudo_epo2._data)
print("Pseudo-dyad created by shuffling epochs within the same dyad.")

pseudo_epo2.plot()

In [ ]:
def create_surrogate_data_fft(epochs):
    """
    Generate surrogate data by shuffling phase information in the frequency domain
    while preserving amplitude information, then reconstructing via inverse FFT.
    This breaks true synchrony while preserving frequency characteristics.
    """
    surrogate_epochs = epochs.copy()
    
    for i in range(len(surrogate_epochs._data)):
        for j in range(surrogate_epochs._data.shape[1]):
            # Get the signal for this epoch and channel
            signal = surrogate_epochs._data[i, j, :]
            
            # Apply FFT
            fft_signal = fft.rfft(signal)
            
            # Extract amplitude and phase
            amplitude = np.abs(fft_signal)
            original_phase = np.angle(fft_signal)
            
            # Shuffle the original phases instead of generating random ones
            shuffled_phase = original_phase.copy()
            np.random.shuffle(shuffled_phase)
            
            # Reconstruct the FFT with original amplitude but shuffled phase
            new_fft_signal = amplitude * np.exp(1j * shuffled_phase)
            
            # Apply inverse FFT and take the real part to get the surrogate time series
            surrogate_epochs._data[i, j, :] = fft.irfft(new_fft_signal, n=len(signal))
    
    return surrogate_epochs

# Create surrogate epochs using FFT phase randomization
surrogate_epo1 = epo1_clean.copy()  # Keep participant 1 unchanged
surrogate_epo2 = create_surrogate_data_fft(epo2_clean)
print("Surrogate data generated using FFT phase randomization for participant 2.")

surrogate_epo2.plot()

## 3. Advanced Connectivity Analysis

We now compute the analytic signals and connectivity metrics using the HyPyP analyses functions for:
- The real (cleaned) dyad.
- The pseudo-dyad.
- The surrogate data.

We define frequency bands (e.g., two alpha sub-bands) and process the data accordingly. The connectivity values are calculated but not normalized, to preserve the original scale of synchronization.

In [ ]:
# Connectivity Analysis for Real, Pseudo, and Surrogate Conditions

# Extract the sampling rate from the epochs (assumes both participants share the same sfreq)
sampling_rate = epo1_clean.info['sfreq']

# Define frequency bands as an OrderedDict
freq_bands = OrderedDict({
    'Alpha-Low': [7.5, 11],
    'Alpha-High': [11.5, 13]
})

# Prepare data arrays for each condition
data_real   = np.array([epo1_clean.get_data(copy=True), epo2_clean.get_data(copy=True)])
data_pseudo = np.array([pseudo_epo1.get_data(copy=True), pseudo_epo2.get_data(copy=True)])
data_surrog = np.array([surrogate_epo1.get_data(copy=True), surrogate_epo2.get_data(copy=True)])

# Compute analytic signals using FIR filtering and the Hilbert transform
complex_real   = analyses.compute_freq_bands(
    data_real, sampling_rate, freq_bands,
    filter_length=int(sampling_rate),
    l_trans_bandwidth=5.0,
    h_trans_bandwidth=5.0
)

complex_pseudo = analyses.compute_freq_bands(
    data_pseudo, sampling_rate, freq_bands,
    filter_length=int(sampling_rate),
    l_trans_bandwidth=5.0,
    h_trans_bandwidth=5.0
)

complex_surrog = analyses.compute_freq_bands(
    data_surrog, sampling_rate, freq_bands,
    filter_length=int(sampling_rate),
    l_trans_bandwidth=5.0,
    h_trans_bandwidth=5.0
)

# Compute connectivity using the circular correlation ('ccorr') metric and average across epochs
conn_real   = analyses.compute_sync(complex_real,   mode='ccorr', epochs_average=True)
conn_pseudo = analyses.compute_sync(complex_pseudo, mode='ccorr', epochs_average=True)
conn_surrog = analyses.compute_sync(complex_surrog, mode='ccorr', epochs_average=True)

# Determine the number of channels per participant
n_ch = len(epo1_clean.info['ch_names'])

# Slice connectivity matrices to extract inter-brain connectivity
alpha_low_real   = conn_real[0, 0:n_ch, n_ch:2*n_ch]
alpha_low_pseudo = conn_pseudo[0, 0:n_ch, n_ch:2*n_ch]
alpha_low_surrog = conn_surrog[0, 0:n_ch, n_ch:2*n_ch]

# Use raw connectivity values without normalization
C_real   = alpha_low_real
C_pseudo = alpha_low_pseudo
C_surrog = alpha_low_surrog

In [ ]:
# 2D Visualization of Real Dyad Connectivity
viz.viz_2D_topomap_inter(epo1_clean, epo2_clean, C_real, threshold='auto', steps=10, lab=True)

In [ ]:
# 2D Visualization of Pseudo-Dyad Connectivity
viz.viz_2D_topomap_inter(pseudo_epo1, pseudo_epo2, C_pseudo, threshold='auto', steps=10, lab=True)

In [ ]:
# 2D Visualization of Surrogate Data Connectivity

viz.viz_2D_topomap_inter(surrogate_epo1, surrogate_epo2, C_surrog, threshold='auto', steps=10, lab=True)

## 4. Discussion and Future Directions

- **Loading Preprocessed Data:**  
  Loading preprocessed data from a previous notebook streamlines the workflow, allowing for faster iteration and analysis.

- **Control Conditions:**  
  Pseudo-dyads (epoch shuffling) and surrogate data (phase randomization via FFT) help determine whether the observed synchrony is interaction-driven or an artifact of shared stimuli or environmental factors.

- **Signal Visualization:**  
  Visualizing the original and modified signals helps validate that our control conditions preserve the general characteristics of EEG data while disrupting the temporal synchrony.

- **Connectivity Metrics:**  
  In this notebook, we used circular correlation (`ccorr`) without normalization, retaining the original scale of synchronization. This allows for direct comparison between conditions.

- **Statistical Analysis:**  
  Subsequent analyses could include statistical comparisons (e.g., permutation tests) between conditions to quantify significant differences.

This notebook highlights how careful experimental design, robust preprocessing, and control conditions contribute to valid hyperscanning analyses.

## Conclusion

In this improved notebook, we have:
- Loaded dyads and applied additional ICA preprocessing to remove artifacts.
- Generated pseudo-dyads and surrogate data as control conditions.
- Computed and normalized connectivity metrics (using circular correlation) for all conditions.
- Visualized the inter-brain connectivity using 2D topographic maps.

These steps lay the groundwork for further investigations into inter-brain synchrony and more advanced statistical analyses.